In [14]:
import gtfs_kit as gk
import pandas as pd

feed = gk.read_feed("../data/gtfs.zip", dist_units="km")
new_feed = gk.read_feed("../data/gtfs.zip", dist_units="km")

In [5]:
feed.stops.dtypes

stop_id       string
stop_name     string
stop_lat     float64
stop_lon     float64
dtype: object

In [6]:
new_stops = [
    ["Bydgoszcz Główna SKM", 53.135296, 17.991672],
    ["Okole SKM", 53.134701, 17.966064],
    ["Wilczak SKM", 53.129897, 17.956174],
    ["Błonie SKM", 53.118294, 17.951722],
    ["Port Lotniczy SKM", 53.09382, 17.99766],
    ["Glinki SKM", 53.095220, 18.044415],
    ["Kapuściska SKM", 53.101897, 18.062957],
    ["Stomil SKM", 53.116273, 18.100124],
    ["Wschód SKM", 53.127983, 18.082905],
    ["Armii Krajowej SKM", 53.139466, 18.038510],
    ["Gdańska SKM", 53.144187, 18.016905]
]

new_stops_df = pd.DataFrame(new_stops, columns=["stop_name", "stop_lat", "stop_lon"])
new_stops_df["stop_id"] = ["new_stop_" + str(i) for i in range(len(new_stops_df))]


In [23]:
new_feed.stops = pd.concat([feed.stops, new_stops_df], ignore_index=True)
new_feed.stops.tail()

,stop_id,stop_name,stop_lat,stop_lon
1192,new_stop_6,Kapuściska SKM,53.101897,18.062957
1193,new_stop_7,Stomil SKM,53.116273,18.100124
1194,new_stop_8,Wschód SKM,53.127983,18.082905
1195,new_stop_9,Armii Krajowej SKM,53.139466,18.038510
1196,new_stop_10,Gdańska SKM,53.144187,18.016905


In [27]:
stop_transition_times = [
    ("Bydgoszcz Główna SKM", "Okole SKM", 2),
    ("Okole SKM", "Wilczak SKM", 1,),
    ("Wilczak SKM", "Błonie SKM", 2),
    ("Błonie SKM", "Port Lotniczy SKM", 6),
    ("Port Lotniczy SKM", "Glinki SKM", 4),
    ("Glinki SKM", "Kapuściska SKM", 1),
    ("Kapuściska SKM", "Stomil SKM", 3),
    ("Stomil SKM", "Wschód SKM", 3),
    ("Wschód SKM", "Armii Krajowej SKM", 3),
    ("Armii Krajowej SKM", "Gdańska SKM", 2),
    ("Gdańska SKM", "Bydgoszcz Główna SKM", 3)
]

In [8]:
routes = feed.routes
routes.head()

,agency_id,route_id,route_short_name,route_long_name,route_type,route_color,route_text_color,route_sort_order
0,0,11,11,Bielawy — Niepodległości,0,DF8B95,000000,111
1,0,2,2,Kapuściska — Las Gdański P+R,0,000691,FFFFFF,102
2,0,3,3,Łoskoń — Wilczak,0,FF0000,FFFFFF,103
3,0,31N,31N,Łoskoń — Dworzec Leśne,3,000000,FFFFFF,300
4,0,32N,32N,Tatrzańskie — Błonie,3,000000,FFFFFF,301


In [10]:
new_routes = [
    (0, "SKM1", "SKM1", "SKM1", 0, "FF0000", "FFFFFF", 1),
]
new_routes_df = pd.DataFrame(new_routes, columns=["agency_id", "route_id", "route_short_name", "route_long_name", "route_type", "route_color", "route_text_color", "route_sort_order"])

In [16]:
new_feed.routes = pd.concat([feed.routes, new_routes_df], ignore_index=True)
new_feed.routes.tail()

,agency_id,route_id,route_short_name,route_long_name,route_type,route_color,route_text_color,route_sort_order
56,0,98,98,Przylesie P+R — Bożenkowo,3,0044BB,FFFFFF,242
57,0,99,99,Plac Kościelickich — Nowa Wieś Wielka,3,0044BB,FFFFFF,243
58,0,ZaT,ZaT,Rondo Toruńskie — Plac Kościeleckich,3,000000,FFFFFF,150
59,0,ZaT6,ZaT6,Rondo Toruńskie — Łęgnowo,3,000000,FFFFFF,150
60,0,SKM1,SKM1,SKM1,0,FF0000,FFFFFF,1


In [18]:
feed.trips.head()

,route_id,service_id,trip_id,trip_headsign,block_id,block_short_name
0,68,2026-07-20:68:4,2026-07-20:875086,Dworzec Leśne,2026-07-20:44602,3
1,68,2026-07-20:68:4,2026-07-20:875087,Park Przemysłowy - Exploseum,2026-07-20:44602,3
2,68,2026-07-20:68:4,2026-07-20:875088,Dworzec Leśne,2026-07-20:44602,3
3,68,2026-07-20:68:4,2026-07-20:875089,Glinki,2026-07-20:44602,3
4,68,2026-07-20:68:4,2026-07-20:875090,Dworzec Leśne,2026-07-20:44602,3


In [ ]:
from time import time

def format_seconds_as_hhmmss(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}"


def construct_new_stop_times(trip_id, stop_sequence, time_start, transition_times):
    stop_times = []
    arrival_time = time_start
    for i in range(len(stop_sequence)):
        departure_time = arrival_time + 30
        stop_id = new_feed.stops.loc[new_feed.stops["stop_name"] == stop_sequence[i], "stop_id"].values[0]
        stop_times.append((trip_id,
                           format_seconds_as_hhmmss(arrival_time),
                           format_seconds_as_hhmmss(departure_time),
                           stop_id,
                           i+1))
        if i < len(stop_sequence) - 1:
            transition_time = next((t[2] for t in transition_times if t[0] == stop_sequence[i] and t[1] == stop_sequence[i+1]), None)
            if transition_time is not None:
                arrival_time = departure_time + transition_time * 60
    return stop_times


def construct_trips(route_id, time_start: time, time_end: time, freq, stop_sequence, transition_times):
    trips = []
    stops = []
    dep_time = time_start
    i = 0
    while dep_time < time_end:
        trip_id = f"{route_id}_trip_{i+1}"
        trips.append((route_id, trip_id, trip_id, route_id, trip_id, trip_id))
        new_stops = construct_new_stop_times(trip_id, stop_sequence, dep_time, transition_times)
        i += 1
        dep_time += freq
        stops.extend(new_stops)
    return trips, stops




In [37]:
new_trip_stops = [
    "Bydgoszcz Główna SKM",
    "Okole SKM",
    "Wilczak SKM",
    "Błonie SKM",
    "Port Lotniczy SKM",
    "Glinki SKM",
    "Kapuściska SKM",
    "Stomil SKM",
    "Wschód SKM",
    "Armii Krajowej SKM",
    "Gdańska SKM",
    "Bydgoszcz Główna SKM"
]

mins = 60
hours = 60 * mins
new_trips, new_stop_times = construct_trips("SKM1", 6*hours, 22*hours, 15*mins, new_trip_stops, stop_transition_times)
new_trips_df = pd.DataFrame(new_trips, columns=["route_id", "service_id", "trip_id", "trip_headsign", "block_id", "block_short_name"])
new_stop_times_df = pd.DataFrame(new_stop_times, columns=["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"])
new_trips_df.head()

,route_id,service_id,trip_id,trip_headsign,block_id,block_short_name
0,SKM1,SKM1_trip_1,SKM1_trip_1,SKM1,SKM1_trip_1,SKM1_trip_1
1,SKM1,SKM1_trip_2,SKM1_trip_2,SKM1,SKM1_trip_2,SKM1_trip_2
2,SKM1,SKM1_trip_3,SKM1_trip_3,SKM1,SKM1_trip_3,SKM1_trip_3
3,SKM1,SKM1_trip_4,SKM1_trip_4,SKM1,SKM1_trip_4,SKM1_trip_4
4,SKM1,SKM1_trip_5,SKM1_trip_5,SKM1,SKM1_trip_5,SKM1_trip_5


In [38]:
new_stop_times_df.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence
0,SKM1_trip_1,06:00:00,06:00:30,new_stop_0,1
1,SKM1_trip_1,06:02:30,06:03:00,new_stop_1,2
2,SKM1_trip_1,06:04:00,06:04:30,new_stop_2,3
3,SKM1_trip_1,06:06:30,06:07:00,new_stop_3,4
4,SKM1_trip_1,06:13:00,06:13:30,new_stop_4,5


In [39]:
new_feed.trips = pd.concat([feed.trips, new_trips_df], ignore_index=True)
new_feed.stop_times = pd.concat([feed.stop_times, new_stop_times_df], ignore_index=True)

In [40]:
import shutil
shutil.copytree("../data", "../data_new", dirs_exist_ok=True)



feed.to_file("../data_new/gtfs.zip")